### Plot BLASTp scores v. composite scores
### Julian Moran
### 2026-08-28

In [2]:
import boto3
import glob
import logging
import math
import os
import requests
import s3fs
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [3]:
# ============================================================
#       Args
# ============================================================

# API endpoints
ENDPOINT_UNIPROT = "https://rest.uniprot.org/uniprotkb/"
ENDPOINT_UNIPROT_SEARCH = "https://rest.uniprot.org/uniprotkb/search"

# MinIO
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_COMPOSITE = "composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/_SUCCESS',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/part-00000-275fe83c-840b-476e-bf63-c47165847863-c000.snappy.parquet']

In [4]:
# ============================================================
#       In
# ============================================================

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_COMPOSITE}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)
df_comp_score

defense_uniprot_ac,human_entryId,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria
str,str,f64,f64,f32,f32,f32,f32,f32
"""A0A5C5QGP9""","""A0A024R9P6""",0.2646,0.000087,null,null,null,null,83.480003
"""A0A2R3IRC4""","""A0A0D9SF92""",0.8315,1.2070e-7,0.93,0.8169,0.1028,83.199997,77.449997
"""A0A4D8PF33""","""A0A140VK70""",0.2883,0.003094,15.01,0.2908,0.1539,80.290001,83.050003
"""A0A7S9D461""","""A0A140VK70""",0.8986,5.6920e-19,2.18,0.535,0.7678,80.290001,89.629997
"""A0A2K9LJD6""","""A0A1B0GVC6""",0.2795,0.007862,10.88,0.2428,0.29,67.610001,91.379997
…,…,…,…,…,…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""",0.6409,0.000001,4.93,0.6556,0.327,84.879997,82.360001
"""A0A7D6CQE3""","""Q9H4E3""",0.5814,0.000001,5.77,0.6733,0.3337,84.879997,75.230003
"""A0A5P3ALB7""","""Q9H4E3""",0.8713,1.2930e-15,2.68,0.739,0.4786,84.879997,89.019997


In [9]:
# ============================================================
#       Annotate with protein sequences
# ============================================================

def get_uniprot_sequence(
    accession: str,
    endpoint_prefix: str
) -> str:
    url = f"{endpoint_prefix}/{accession}.fasta"
    response = requests.get(url)
    response.raise_for_status()

    sequence = "".join(response.text.splitlines()[1:])
    return sequence

def batch_uniprot_sequences(
    accessions: pl.Series,
    endpoint_prefix: str,
) -> pl.DataFrame:
    col_accessions = accessions.name
    accessions = accessions.to_list()
    query = " OR ".join(f"accession:{acc}" for acc in accessions)

    response = requests.get(
        endpoint_prefix,
        params={
            "query": query,
            "format": "fasta",
            "size": len(accessions),
        },
    )
    response.raise_for_status()
    records = response.text.strip().split("\n>")
    parsed_accessions = []
    sequences = []

    logger.info(
        f"Requested: {len(accessions)}, "
        f"Returned: {len(records)}"
    )
    for record in records:
        record = record.lstrip(">")

        lines = record.splitlines()
        header = lines[0]
        sequence = "".join(lines[1:])

        if "|" not in header:
            logger.warning(f"Unexpected FASTA header: {header!r}")
            continue

        parsed_accessions.append(header.split("|")[1])
        sequences.append(sequence)

    return pl.DataFrame({
        "accession": parsed_accessions,
        "sequence": sequences,
    })

def build_seq_lookup(
    accessions: pl.Series,
    endpoint_prefix: str,
    batch_size: int = 100,
    wait_time: float = 1.0,
) -> pl.DataFrame:
    batches = []

    for i in range(0, len(accessions), batch_size):
        batches.append(
            batch_uniprot_sequences(
                accessions=accessions[i:i + batch_size],
                endpoint_prefix=endpoint_prefix,
            )
        )
        logger.info(f"Received {i + batch_size} sequences of {len(accessions)}")
        time.sleep(wait_time)
    return pl.concat(batches)

hs_accessions_unq = df_comp_score["human_entryId"].unique()

hs_seq_lookup = build_seq_lookup(
    accessions=hs_accessions_unq,
    endpoint_prefix=ENDPOINT_UNIPROT_SEARCH,
)
hs_seq_lookup

INFO:__main__:Requested: 100, Returned: 65
INFO:__main__:Received 100 sequences of 15572
INFO:__main__:Requested: 100, Returned: 66
INFO:__main__:Received 200 sequences of 15572


KeyboardInterrupt: 

In [ ]:
def batch_uniprot_sequences(
    accessions: pl.Series,
    endpoint_prefix: str,
) -> pl.DataFrame:
    accessions = accessions.to_list()
    query = " OR ".join(f"accession:{acc}" for acc in accessions)

    response = requests.get(
        endpoint_prefix,
        params={
            "query": query,
            "format": "fasta",
            "size": len(accessions),
        },
    )
    response.raise_for_status()
    records = response.text.strip().split("\n>")
    parsed_accessions = []
    sequences = []

    logger.info(
        f"UniProtKB requested: {len(accessions)}, "
        f"UniProtKB returned: {len(records)}"
    )

    for record in records:
        record = record.lstrip(">")
        lines = record.splitlines()
        header = lines[0]
        sequence = "".join(lines[1:])

        if "|" not in header:
            logger.warning(f"Unexpected FASTA header: {header!r}")
            continue

        parsed_accessions.append(header.split("|")[1])
        sequences.append(sequence)

    return pl.DataFrame({
        "accession": parsed_accessions,
        "sequence": sequences,
    })

def map_uniprotkb_to_uniparc(
    accessions: pl.Series,
    wait_time: float = 1.0
) -> pl.DataFrame:

    mapping_response = requests.post(
        "https://rest.uniprot.org/idmapping/run",
        data={
            "from": "UniProtKB_AC-ID",
            "to": "UniParc",
            "ids": ",".join(accessions.to_list()),
        },
    )
    mapping_response.raise_for_status()
    mapping_job_id = mapping_response.json()["jobId"]

    while True:
        status_response = requests.get(
            f"https://rest.uniprot.org/idmapping/status/{mapping_job_id}"
        )
        status_response.raise_for_status()
        status = status_response.json()
        if status.get("jobStatus") == "RUNNING":
            time.sleep(wait_time)
            continue
        if status.get("jobStatus") == "FAILED":
            raise RuntimeError(
                f"UniProt ID mapping failed: {status}"
            )
        break

    response = requests.get(
        f"https://rest.uniprot.org/idmapping/uniparc/results/{mapping_job_id}",
        params={"format": "tsv"},
    )
    response.raise_for_status()
    mapping_lines = response.text.strip().splitlines()

    # Skip header.
    mapping = [
        line.split("\t")[:2]
        for line in mapping_lines[1:]
    ]
    uniparc_mapping = pl.DataFrame(
        mapping,
        schema=["uniprot_accession", "uniparc_accession"],
        orient="row",
    )
    return uniparc_mapping


def build_seq_lookup(
    accessions: pl.Series,
    uniprot_endpoint: str,
    uniparc_endpoint: str,
    batch_size: int = 100,
    wait_time: float = 1.0,
) -> pl.DataFrame:
    # Find sequences in UniProtKB
    uniprot_batches = []
    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i + batch_size]
        uniprot_batches.append(
            batch_uniprot_sequences(
                accessions=batch,
                endpoint_prefix=uniprot_endpoint,
            )
        )
        logger.info(
            f"UniProtKB: processed "
            f"{min(i + batch_size, len(accessions))} "
            f"of {len(accessions)} accessions"
        )
        time.sleep(wait_time)
    uniprotkb_sequences = pl.concat(uniprot_batches)

    # Find acessions for sequences not in UniProtKB
    found_accessions = uniprotkb_sequences["accession"]
    missing_accessions = accessions.filter(
        ~accessions.is_in(found_accessions.implode())
    )
    logger.info(
        f"UniProtKB missing: {len(missing_accessions)} "
        f"of {len(accessions)} accessions"
    )

    # Map missing UniProtKB accessions to UniParc accessions.
    df_uniparc_mapping = map_uniprotkb_to_uniparc(missing_accessions)
    return df_uniparc_mapping

    # # Retrieve the UniParc sequences.
    # uniparc_batches = []
    # for i in range(0, len(uniparc_mapping), batch_size):
    #     batch = uniparc_mapping[i:i + batch_size]
    #     sequences = []

    #     for row in batch.iter_rows(named=True):
    #         response = requests.get(
    #             f"{uniparc_endpoint}/{row['uniparc_accession']}.fasta"
    #         )
    #         response.raise_for_status()

    #         lines = response.text.strip().splitlines()
    #         sequence = "".join(lines[1:])

    #         sequences.append({
    #             "accession": row["uniprot_accession"],
    #             "sequence": sequence,
    #         })

    #     uniparc_batches.append(pl.DataFrame(sequences))

    #     logger.info(
    #         f"UniParc: processed "
    #         f"{min(i + batch_size, len(uniparc_mapping))} "
    #         f"of {len(uniparc_mapping)} mapped accessions"
    #     )

    #     time.sleep(wait_time)

    # if uniparc_batches:
    #     uniparc_sequences = pl.concat(uniparc_batches)
    #     return pl.concat([uniprot_sequences, uniparc_sequences])
    # return uniprot_sequences

tmp = df_comp_score["human_entryId"].unique()

result = build_seq_lookup(
    accessions=tmp[:100],
    uniprot_endpoint=ENDPOINT_UNIPROT_SEARCH,
    uniparc_endpoint=""
)
result

INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 64
INFO:__main__:UniProtKB: processed 100 of 100 accessions
INFO:__main__:UniProtKB missing: 36 of 100 accessions


From	Entry	Organisms	UniProtKB	First seen	Last seen	Length


uniprot_accession,uniparc_accession
str,str
"""B7SEW4""","""UPI000188CA54"""
"""A0A024R9G1""","""UPI0000EE3142"""
"""F8R159""","""UPI0001E5A267"""
"""A0A0E3DC77""","""UPI0000062349"""
"""C9WER6""","""UPI0001B9FAE3"""
…,…
"""A0A024R567""","""UPI000013E13D"""
"""A0A024R6E7""","""UPI000013D23C"""
"""E2DH79""","""UPI000008AFE7"""
